In [13]:
# !pip install pandas

In [16]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

%matplotlib inline

In [17]:
df = pd.read_csv('Dataset/phishing_site_urls.csv')

In [18]:
df.head()

,URL,Label
0,nobell.it/70ffb52d079109dca5664cce6f317373782/...,bad
1,www.dghjdgf.com/paypal.co.uk/cycgi-bin/webscrc...,bad
2,serviciosbys.com/paypal.cgi.bin.get-into.herf....,bad
3,mail.printakid.com/www.online.americanexpress....,bad
4,thewhiskeydregs.com/wp-content/themes/widescre...,bad


In [19]:
df.shape

(549346, 2)

In [20]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 549346 entries, 0 to 549345
Data columns (total 2 columns):
 #   Column  Non-Null Count   Dtype
---  ------  --------------   -----
 0   URL     549346 non-null  str  
 1   Label   549346 non-null  str  
dtypes: str(2)
memory usage: 36.9 MB


In [21]:
df.isnull().sum()

URL      0
Label    0
dtype: int64

In [22]:
df.Label.value_counts()

Label
good    392924
bad     156422
Name: count, dtype: int64

In [23]:
import re
import tldextract
from urllib.parse import urlparse

In [15]:
!pip install tldextract

In [24]:
def extract_features(url):

    url = str(url).strip().lower()

    # Add protocol for parsing if missing
    if not url.startswith(("http://", "https://")):
        url_to_parse = "http://" + url
    else:
        url_to_parse = url

    # Parse URL safely
    try:
        parsed = urlparse(url_to_parse)

        hostname = parsed.hostname or ""
        path = parsed.path or ""
        query = parsed.query or ""

    except:
        hostname = ""
        path = ""
        query = ""

    # Extract domain information
    extracted = tldextract.extract(url_to_parse)

    subdomain = extracted.subdomain or ""
    domain = extracted.domain or ""
    suffix = extracted.suffix or ""

    # Suspicious words
    suspicious_words = [
        "login", "signin", "verify", "verification",
        "secure", "account", "update", "password",
        "confirm", "bank", "payment", "wallet",
        "authenticate", "security", "support"
    ]

    suspicious_count = sum(
        word in url
        for word in suspicious_words
    )

    # Check if hostname is an IP address
    ip_pattern = r"^(?:\d{1,3}\.){3}\d{1,3}$"

    is_ip_address = int(
        bool(re.match(ip_pattern, hostname))
    )

    # Number of subdomains
    subdomain_count = (
        subdomain.count(".") + 1
        if subdomain else 0
    )

    return {

        # Existing/basic features
        "url_length": len(url),
        "hostname_length": len(hostname),
        "path_length": len(path),

        "domain_length": len(domain),
        "subdomain_length": len(subdomain),
        "subdomain_count": subdomain_count,

        "dot_count": url.count("."),
        "hyphen_count": url.count("-"),
        "underscore_count": url.count("_"),
        "digit_count": sum(c.isdigit() for c in url),

        "slash_count": url.count("/"),
        "question_count": url.count("?"),
        "equal_count": url.count("="),
        "at_count": url.count("@"),

        "https": int(url.startswith("https://")),

        "suspicious_word_count": suspicious_count,

        "domain_has_digit": int(
            any(c.isdigit() for c in domain)
        ),

        "domain_has_hyphen": int(
            "-" in domain
        ),

        # New features
        "query_length": len(query),

        "has_ip_address": is_ip_address,

        "double_slash_path": int(
            "//" in path
        ),

        "percent_count": url.count("%"),

        "colon_count": url.count(":"),

        "www_count": url.count("www"),

        "has_punycode": int(
            "xn--" in hostname
        )
    }

In [25]:
url_features = df["URL"].apply(extract_features)

X = pd.DataFrame(url_features.tolist())

print(X.head())
print("X shape:", X.shape)

   url_length  hostname_length  path_length  domain_length  subdomain_length  \
0         225                9          125              6                 0   
1          81               15           66              7                 3   
2         177               16          161             12                 0   
3          60               18           42              9                 4   
4         116               19           60             15                 0   

   subdomain_count  dot_count  hyphen_count  underscore_count  digit_count  \
0                0          6             4                 4           58   
1                1          5             2                 1            1   
2                0          7             1                 0           47   
3                1          6             0                 0            0   
4                0          1             1                 0           21   

   ...  suspicious_word_count  domain_has_digit  d

In [26]:
y = df["Label"]

print("Original labels:")
print(y.value_counts())

y = y.map({
    "bad": 0,
    "good": 1
})

print("\nEncoded labels:")
print(y.value_counts())

Original labels:
Label
good    392924
bad     156422
Name: count, dtype: int64

Encoded labels:
Label
1    392924
0    156422
Name: count, dtype: int64


In [27]:
print("Missing values in y:", y.isnull().sum())

Missing values in y: 0


In [28]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("X_train:", X_train.shape)
print("X_test:", X_test.shape)

print("\ny_train distribution:")
print(y_train.value_counts())

print("\ny_test distribution:")
print(y_test.value_counts())

X_train: (439476, 25)
X_test: (109870, 25)

y_train distribution:
Label
1    314339
0    125137
Name: count, dtype: int64

y_test distribution:
Label
1    78585
0    31285
Name: count, dtype: int64


In [29]:
from sklearn.ensemble import RandomForestClassifier

model = RandomForestClassifier(
    n_estimators=300,
    random_state=42,
    n_jobs=-1,
    class_weight="balanced"
)

model.fit(X_train, y_train)

print("Model training completed!")

Model training completed!


In [30]:
y_pred = model.predict(X_test)

print("Predictions completed!")

Predictions completed!


In [31]:
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    precision_score,
    recall_score,
    f1_score
)

accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", round(accuracy * 100, 2), "%")

print("\nPrecision:", round(
    precision_score(y_test, y_pred) * 100, 2
), "%")

print("Recall:", round(
    recall_score(y_test, y_pred) * 100, 2
), "%")

print("F1 Score:", round(
    f1_score(y_test, y_pred) * 100, 2
), "%")

print("\nClassification Report:")
print(classification_report(y_test, y_pred))

print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))

Accuracy: 92.93 %

Precision: 96.05 %
Recall: 93.98 %
F1 Score: 95.0 %

Classification Report:
              precision    recall  f1-score   support

           0       0.86      0.90      0.88     31285
           1       0.96      0.94      0.95     78585

    accuracy                           0.93    109870
   macro avg       0.91      0.92      0.91    109870
weighted avg       0.93      0.93      0.93    109870


Confusion Matrix:
[[28248  3037]
 [ 4731 73854]]


In [32]:
false_negatives = (
    (y_test == 0) &
    (y_pred == 1)
)

print(
    "Dangerous False Negatives:",
    false_negatives.sum()
)

print(
    "False Negative Rate:",
    round(
        false_negatives.sum() / (y_test == 0).sum() * 100,
        2
    ),
    "%"
)

Dangerous False Negatives: 3037
False Negative Rate: 9.71 %


In [35]:
# Get prediction probabilities

probabilities = model.predict_proba(X_test)

print("Classes:", model.classes_)
print("Probability shape:", probabilities.shape)

Classes: [0 1]
Probability shape: (109870, 2)


In [36]:
import numpy as np
from sklearn.metrics import confusion_matrix

# Probability of GOOD class (class 1)
good_probability = probabilities[:, 1]

thresholds = [0.50, 0.55, 0.60, 0.65, 0.70, 0.75]

results = []

for threshold in thresholds:

    # Classify as GOOD only when probability is high enough
    threshold_predictions = np.where(
        good_probability >= threshold,
        1,
        0
    )

    cm = confusion_matrix(y_test, threshold_predictions)

    # Actual BAD predicted as GOOD
    bad_predicted_good = cm[0, 1]

    # Actual GOOD predicted as BAD
    good_predicted_bad = cm[1, 0]

    # Percentage of BAD URLs correctly detected
    bad_detection_recall = (
        cm[0, 0] /
        (cm[0, 0] + cm[0, 1])
        * 100
    )

    results.append({
        "Threshold": threshold,
        "Bad predicted as Good": bad_predicted_good,
        "Good predicted as Bad": good_predicted_bad,
        "Bad Detection Recall (%)": round(
            bad_detection_recall, 2
        )
    })

results_df = pd.DataFrame(results)

print(results_df)

   Threshold  Bad predicted as Good  Good predicted as Bad  \
0       0.50                   3041                   4714   
1       0.55                   2755                   5491   
2       0.60                   2435                   6462   
3       0.65                   2117                   7659   
4       0.70                   1769                   9272   
5       0.75                   1440                  10958   

   Bad Detection Recall (%)  
0                     90.28  
1                     91.19  
2                     92.22  
3                     93.23  
4                     94.35  
5                     95.40  


In [37]:
# -----------------------------------------
# FINAL EVALUATION USING THRESHOLD = 0.65
# -----------------------------------------

from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix
)

FINAL_THRESHOLD = 0.65

# Predict GOOD only when the probability of GOOD
# is equal to or greater than 0.65
final_y_pred = np.where(
    good_probability >= FINAL_THRESHOLD,
    1,
    0
)

print("Final Threshold:", FINAL_THRESHOLD)

print(
    "\nAccuracy:",
    round(accuracy_score(y_test, final_y_pred) * 100, 2),
    "%"
)

print("\nClassification Report:")
print(
    classification_report(
        y_test,
        final_y_pred
    )
)

print("\nConfusion Matrix:")
print(
    confusion_matrix(
        y_test,
        final_y_pred
    )
)

# Dangerous phishing URLs predicted as legitimate
final_false_negatives = (
    (y_test == 0) &
    (final_y_pred == 1)
)

print(
    "\nDangerous False Negatives:",
    final_false_negatives.sum()
)

print(
    "False Negative Rate:",
    round(
        final_false_negatives.sum()
        / (y_test == 0).sum()
        * 100,
        2
    ),
    "%"
)

Final Threshold: 0.65

Accuracy: 91.1 %

Classification Report:
              precision    recall  f1-score   support

           0       0.79      0.93      0.86     31285
           1       0.97      0.90      0.94     78585

    accuracy                           0.91    109870
   macro avg       0.88      0.92      0.90    109870
weighted avg       0.92      0.91      0.91    109870


Confusion Matrix:
[[29168  2117]
 [ 7659 70926]]

Dangerous False Negatives: 2117
False Negative Rate: 6.77 %


In [33]:
feature_importance = pd.DataFrame({
    "Feature": X.columns,
    "Importance": model.feature_importances_
}).sort_values(
    by="Importance",
    ascending=False
)

print(feature_importance)

                  Feature  Importance
9             digit_count    0.120066
2             path_length    0.119914
0              url_length    0.097280
3           domain_length    0.090098
15  suspicious_word_count    0.083278
10            slash_count    0.078522
6               dot_count    0.076278
1         hostname_length    0.066800
7            hyphen_count    0.066340
4        subdomain_length    0.036593
18           query_length    0.030214
8        underscore_count    0.029726
16       domain_has_digit    0.025654
5         subdomain_count    0.016860
12            equal_count    0.014776
17      domain_has_hyphen    0.011018
23              www_count    0.010810
11         question_count    0.007803
19         has_ip_address    0.006730
21          percent_count    0.004740
22            colon_count    0.003531
13               at_count    0.001732
20      double_slash_path    0.000975
24           has_punycode    0.000262
14                  https    0.000001


In [38]:
print("Number of features used by the model:", len(model.feature_names_in_))

print("\nFeature names:")
print(model.feature_names_in_)

Number of features used by the model: 25

Feature names:
['url_length' 'hostname_length' 'path_length' 'domain_length'
 'subdomain_length' 'subdomain_count' 'dot_count' 'hyphen_count'
 'underscore_count' 'digit_count' 'slash_count' 'question_count'
 'equal_count' 'at_count' 'https' 'suspicious_word_count'
 'domain_has_digit' 'domain_has_hyphen' 'query_length' 'has_ip_address'
 'double_slash_path' 'percent_count' 'colon_count' 'www_count'
 'has_punycode']


In [34]:
import pickle

with open("phishing_model.pkl", "wb") as file:
    pickle.dump(model, file)

print("Model saved successfully!")

Model saved successfully!
